# XMACE Oscillator Strengths

This notebook shows how to train an X-MACE model to predict **oscillator strengths**

### The core idea

Oscillator strengths are **scalar values per electronic state transition**, just like energies. This means we can exploit X-MACE's existing multi-state energy machinery directly, with no new model architecture required:

1. **Pack all oscillator strengths into the `energy` field** of the XYZ file, with the `REF_energy` field holding the corresponding reference values.
2. **Set `--n_energies`** to the total number of oscillator strength values per frame.
3. Train with `--energy_weight` — the model learns the oscillator strengths as a vector of scalars, exactly as it would learn a multi-state energy vector.
4. At inference time, `calc.results["energy"]` gives you back the predicted oscillator strengths.

Because oscillator strengths are geometry-dependent scalars, the energy readout head (which maps atom-centred features to a sum of scalars) is a natural fit — there is no need for forces, so `--forces_weight=0.0`.

## 1. The Oscillator Strength Data Layout

For a molecule with `n_states` electronic states, the number of distinct **state-to-state transitions** (and thus oscillator strengths) from the ground state is `n_states - 1`. If you have oscillator strengths for all unique pairs, the count is `n_states * (n_states - 1) / 2`.

The key mapping is:

| XYZ field | Content | Shape | When present |
|---|---|---|---|
| `REF_energy` | reference oscillator strengths | `(1, N_OSC)` | Always, in the dataset file |
| `calc.results['energy']` | predicted oscillator strengths | `(N_OSC,)` | At inference time only, from the calculator |

All other fields (`forces`, `smooth_nacs`, `socs`, …) are **not used** — set their loss weights to `0.0` when training.

In [3]:
# --- Derive S0→S1 and S0→S2 oscillator strengths from CASSCF data ---
# Assumptions: REF_energy is in eV; REF_dip_trans is in e·Å; its rows
# are ordered (S0,S1), (S0,S2), (S1,S2).
from pathlib import Path

import ase.io
import numpy as np

INPUT_FILE = Path("/home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_FINAL_annotated.xyz")
OUTPUT_FILE = Path("/home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_osc_strengths.xyz")
EV_PER_HARTREE = 27.211386245988
BOHR_PER_ANGSTROM = 1.8897261254578281
TRANSITION_ROWS = (0, 1)  # S0→S1, S0→S2

frames = ase.io.read(INPUT_FILE, index=":")
oscillator_strengths = []

for frame_index, frame in enumerate(frames):
    energies_ev = np.asarray(frame.info["REF_energy"], dtype=float).reshape(-1)
    transition_dipoles_ea = np.asarray(frame.info["REF_dip_trans"], dtype=float)
    if energies_ev.size < 3 or transition_dipoles_ea.shape != (3, 3):
        raise ValueError(
            f"Frame {frame_index}: expected 3 state energies and REF_dip_trans shape (3, 3), "
            f"got {energies_ev.shape} and {transition_dipoles_ea.shape}"
        )

    gaps_hartree = (energies_ev[[1, 2]] - energies_ev[0]) / EV_PER_HARTREE
    transition_dipoles_au = transition_dipoles_ea[list(TRANSITION_ROWS)] * BOHR_PER_ANGSTROM
    strengths = (2.0 / 3.0) * gaps_hartree * np.sum(transition_dipoles_au**2, axis=1)
    if not np.all(np.isfinite(strengths)) or np.any(strengths < 0.0):
        raise ValueError(f"Frame {frame_index}: invalid oscillator strengths {strengths}")

    frame.info["REF_energy"] = strengths.reshape(1, 2)
    oscillator_strengths.append(strengths)

oscillator_strengths = np.asarray(oscillator_strengths)
ase.io.write(OUTPUT_FILE, frames)

print(f"Written {len(frames)} frames to '{OUTPUT_FILE}'")
print(f"REF_energy shape: {np.asarray(frames[0].info['REF_energy']).shape}")
print(f"S0→S1 oscillator strengths: min={oscillator_strengths[:, 0].min():.8g}, max={oscillator_strengths[:, 0].max():.8g}")
print(f"S0→S2 oscillator strengths: min={oscillator_strengths[:, 1].min():.8g}, max={oscillator_strengths[:, 1].max():.8g}")


Written 3731 frames to '/home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_osc_strengths.xyz'
REF_energy shape: (1, 2)
S0→S1 oscillator strengths: min=0, max=0.60313704
S0→S2 oscillator strengths: min=0, max=0


In [6]:
frames = ase.io.read("/home/lim_yt/X-MACE-sampling/data/A01_ethene/static/A01_ethene_grid_static_CASSCF_osc_strengths.xyz", ":")
print(f"Total frames: {len(frames)}")
print()

# Only REF_energy should be present. We do NOT check for atoms.info['energy']
# because that key does not exist in the dataset - predictions are written
# to calc.results['energy'] at inference time, never to atoms.info.
all_ok = True
for i, frame in enumerate(frames):
    if 'REF_energy' not in frame.info:
        print(f"  Frame {i:4d}  missing REF_energy  ← ERROR")
        all_ok = False
        continue
    ref_shape = np.array(frame.info['REF_energy']).shape
    expected  = (1, N_OSC)
    ok = ref_shape == expected
    if not ok:
        print(f"  Frame {i:4d}  REF_energy={ref_shape}  expected={expected}  ← WRONG")
        all_ok = False

if all_ok:
    print(f"All {len(frames)} frames have REF_energy shape (1, {N_OSC}). Ready to train.")
else:
    print("Some frames have issues — fix them before training.")


Total frames: 3731

All 3731 frames have REF_energy shape (1, 2). Ready to train.
